# Evaluate RAG

## Paths

In [20]:
# find project files

from pathlib import Path


cwd = Path.cwd().resolve()

candidates = []

for path in [cwd, *cwd.parents]:
    candidates.extend([
        path,
        path / "fitness-assistant",
        path / "07-project-example" / "fitness-assistant",
        path / "datatalks" / "llm" / "zoomcamp-llm-2026" / "07-project-example" / "fitness-assistant",
    ])

PROJECT_DIR = None

for candidate in candidates:
    if (candidate / "data" / "data.csv").exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find fitness-assistant/data/data.csv")

COURSE_ROOT = None

for path in [PROJECT_DIR, *PROJECT_DIR.parents]:
    if (path / "pyproject.toml").exists():
        COURSE_ROOT = path
        break

if COURSE_ROOT is None:
    raise FileNotFoundError("Could not find course root with pyproject.toml")

DATA_DIR = PROJECT_DIR / "data"

print("Project dir:", PROJECT_DIR)
print("Data dir:", DATA_DIR)

Project dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant
Data dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant/data


## Packages

In [21]:
# load packages and OpenAI

import json
import os
import random

import pandas as pd
from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI
from tqdm.auto import tqdm


load_dotenv(COURSE_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Add it to the course root .env file.")

openai_client = OpenAI()
MODEL = "gpt-5.4-mini"

MODEL

'gpt-5.4-mini'

## Load data

In [22]:
# load exercises and ground truth questions

csv_path = DATA_DIR / "data.csv"
ground_truth_path = DATA_DIR / "ground-truth-retrieval.csv"

if not ground_truth_path.exists():
    raise FileNotFoundError("Run 03_evaluate_retrieval.ipynb first to create ground-truth-retrieval.csv")

df = pd.read_csv(csv_path)
df_questions = pd.read_csv(ground_truth_path)

documents = df.to_dict(orient="records")
ground_truth = df_questions.to_dict(orient="records")

print("Exercises:", len(documents))
print("Questions:", len(ground_truth))
df_questions.head()

Exercises: 50
Questions: 100


,id,question
0,push-up-001,How do I perform a push-up with proper form?
1,push-up-001,Which muscles does a push-up work?
2,squat-002,How can I make sure my knees stay aligned over...
3,squat-002,How low should I go in a bodyweight squat whil...
4,plank-003,How long should I hold a plank as a beginner?


## Build index

In [23]:
# same minsearch index used in the RAG notebook

index = Index(
    text_fields=[
        "exercise_name",
        "type_of_activity",
        "type_of_equipment",
        "body_part",
        "type",
        "muscle_groups_activated",
        "instructions",
    ],
    keyword_fields=["id"],
)

index.fit(documents)

## Retrieval metrics

In [24]:
# reuse retrieval metrics so the search setup is checked before RAG evaluation


def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt += 1

    return cnt / len(relevance_total)



def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)



def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q["id"]
        results = search_function(q)
        relevance = [d["id"] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Search

In [25]:
# search can use default or tuned boost values


def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10,
    )

    return results

## Tune boosts

In [26]:
# tune boosts again so this notebook can run by itself

random.seed(1)

split_idx = int(len(df_questions) * 0.7)

df_validation = df_questions[:split_idx]
df_test = df_questions[split_idx:]

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

param_ranges = {
    "exercise_name": (0.0, 3.0),
    "type_of_activity": (0.0, 3.0),
    "type_of_equipment": (0.0, 3.0),
    "body_part": (0.0, 3.0),
    "type": (0.0, 3.0),
    "muscle_groups_activated": (0.0, 3.0),
    "instructions": (0.0, 3.0),
}


def simple_optimize(param_ranges, objective_function, n_iterations=20):
    best_params = None
    best_score = float("-inf")

    for _ in range(n_iterations):
        current_params = {}

        for field, (low, high) in param_ranges.items():
            current_params[field] = random.uniform(low, high)

        current_score = objective_function(current_params)

        if current_score > best_score:
            best_score = current_score
            best_params = current_params

    return best_params



def objective(boost_params):
    def search_function(q):
        return minsearch_search(q["question"], boost=boost_params)

    results = evaluate(gt_val, search_function)
    return results["hit_rate"]


best_params = simple_optimize(param_ranges, objective, n_iterations=20)

best_params

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

...

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

{'exercise_name': 0.40309273233720366,
 'type_of_activity': 2.542301210811698,
 'type_of_equipment': 2.291323856929842,
 'body_part': 0.7652070772182651,
 'type': 1.486305261275823,
 'muscle_groups_activated': 1.3484731943662145,
 'instructions': 1.9547789181682889}

## Final search

In [27]:
# this tuned search is used by the RAG function


def search(query):
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=best_params,
        num_results=10,
    )

    return results


evaluate(gt_test, lambda q: search(q["question"]))

  0%|          | 0/30 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.9027777777777778}

## Prompt

In [28]:
# prompt used for the fitness assistant answer

entry_template = """
exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}
""".strip()

prompt_template = """
You're a fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()



def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

## LLM

In [29]:
# OpenAI call used by both RAG and judge


def llm(prompt, model=MODEL):
    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
    )

    return response.output_text

## RAG

In [30]:
# full RAG flow: search, prompt, answer


def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, model=model)

    return answer

## Judge prompt

In [31]:
# judge prompt from the RAG evaluation lesson

prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as 'NON_RELEVANT', 'PARTLY_RELEVANT', or 'RELEVANT'.

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "Provide a brief explanation for your evaluation"
}}
""".strip()

## Test one question

In [32]:
# test the full RAG plus judge flow on one question first

record = ground_truth[0]
question = record["question"]
answer_llm = rag(question)

prompt = prompt2_template.format(
    question=question,
    answer_llm=answer_llm,
)

evaluation = llm(prompt)
evaluation = json.loads(evaluation)

print("Question:", question)
print("Answer:", answer_llm)
evaluation

Question: How do I perform a push-up with proper form?
Answer: To perform a **push-up** with proper form:

- Start in a **high plank** with your **hands slightly wider than shoulder-width**.
- Keep your body in a **straight line from head to heels**.
- **Lower your chest toward the floor** by bending your elbows.
- **Press through your palms** to return to the starting position.
- Keep your **core braced throughout**.


{'Relevance': 'RELEVANT',
 'Explanation': 'The answer directly addresses how to perform a push-up with proper form and provides clear, correct, and relevant steps covering body position, movement, and core engagement.'}

## Sample questions

In [33]:
# keep the sample small first, then increase after the notebook works

sample_size = min(20, len(df_questions))

df_sample = df_questions.sample(n=sample_size, random_state=1)
sample = df_sample.to_dict(orient="records")

sample[:3]

[{'id': 'box-jump-041',
  'question': 'How do I perform a box jump safely with proper landing mechanics?'},
 {'id': 'battle-rope-043',
  'question': 'How do I perform battle rope waves with proper form and breathing?'},
 {'id': 'lat-pulldown-017',
  'question': 'What are common mistakes to avoid when doing the lat pulldown machine exercise?'}]

## Run evaluation

In [ ]:
# this cell calls OpenAI many times: one answer call and one judge call per question

evaluations = []

for record in tqdm(sample):
    question = record["question"]
    answer_llm = rag(question)

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm,
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)

    evaluations.append((record, answer_llm, evaluation))

len(evaluations)

  0%|          | 0/20 [00:00<?, ?it/s]

## Results

In [ ]:
# turn judge output into a table

df_eval = pd.DataFrame(evaluations, columns=["record", "answer", "evaluation"])

df_eval["id"] = df_eval.record.apply(lambda d: d["id"])
df_eval["question"] = df_eval.record.apply(lambda d: d["question"])
df_eval["relevance"] = df_eval.evaluation.apply(lambda d: d["Relevance"])
df_eval["explanation"] = df_eval.evaluation.apply(lambda d: d["Explanation"])

df_eval = df_eval[[
    "id",
    "question",
    "answer",
    "relevance",
    "explanation",
]]

df_eval.head()

,id,question,answer,relevance,explanation
0,box-jump-041,How do I perform a box jump safely with proper...,To perform a box jump safely with proper landi...,RELEVANT,The answer directly addresses how to perform a...
1,battle-rope-043,How do I perform battle rope waves with proper...,To perform **battle rope waves** with proper f...,RELEVANT,The answer directly addresses how to perform b...
2,lat-pulldown-017,What are common mistakes to avoid when doing t...,Common mistakes to avoid on the lat pulldown a...,RELEVANT,The answer directly addresses the question by ...
3,box-jump-041,What muscles does the box jump work and how ca...,"The **box jump** works the **quadriceps, glute...",PARTLY_RELEVANT,The answer correctly identifies the muscles wo...
4,bulgarian-split-squat-047,How low should I go in a Bulgarian split squat...,"In a Bulgarian split squat, lower your body by...",RELEVANT,The answer directly addresses both parts of th...


## Relevance counts

In [ ]:
# final RAG quality summary

df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.95
PARTLY_RELEVANT    0.05
Name: proportion, dtype: float64

## Save evaluation

In [ ]:
# save results for reporting and later comparison

output_path = DATA_DIR / f"rag-eval-{MODEL}.csv"

df_eval.to_csv(output_path, index=False)

output_path

PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant/data/rag-eval-gpt-5.4-mini.csv')